In [4]:
"""
=============================================================
  Time-Series Forecasting Pipeline
  Roll Number : 102303372
  Model       : Custom RNN (last digit = 2, EVEN)
  Dataset 1   : Electric Production (sir's dataset)
  Dataset 2   : International Airline Passengers (my choice)
=============================================================

PARAMETER DERIVATION FROM ROLL NUMBER 102303372:
  Digits            : 1, 0, 2, 3, 0, 3, 3, 7, 2
  Sum of digits     : 1+0+2+3+0+3+3+7+2 = 21
  Last 2 digits     : 72
  First 3 digits    : 102

  window_size        = (sum of digits) mod 10 + 8 = 21 mod 10 + 8 = 1 + 8 = 9
  prediction_horizon = (last 2 digits)  mod 3  + 1 = 72 mod 3  + 1 = 0 + 1 = 1
  hidden_size        = (first 3 digits) mod 16 + 8 = 102 mod 16 + 8 = 6 + 8 = 14

  Last digit = 2 → EVEN → Assigned model: Custom RNN
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math
import os
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# SECTION 0: PERSONALIZED PARAMETERS FROM ROLL NUMBER
# ─────────────────────────────────────────────────────────────

ROLL_NUMBER = "102303372"
digits = [int(d) for d in ROLL_NUMBER]

window_size        = (sum(digits)) % 10 + 8           # = 9
prediction_horizon = (int(ROLL_NUMBER[-2:])) % 3 + 1  # = 1
hidden_size        = (int(ROLL_NUMBER[:3])) % 16 + 8  # = 14

print("=" * 60)
print("  ROLL NUMBER PARAMETER DERIVATION")
print("=" * 60)
print(f"  Roll Number        : {ROLL_NUMBER}")
print(f"  Digits             : {digits}")
print(f"  Sum of digits      : {sum(digits)}")
print(f"  Last 2 digits      : {ROLL_NUMBER[-2:]}")
print(f"  First 3 digits     : {ROLL_NUMBER[:3]}")
print(f"  window_size        = {sum(digits)} mod 10 + 8 = {window_size}")
print(f"  prediction_horizon = {int(ROLL_NUMBER[-2:])} mod 3 + 1 = {prediction_horizon}")
print(f"  hidden_size        = {int(ROLL_NUMBER[:3])} mod 16 + 8 = {hidden_size}")
print(f"  Last digit = {digits[-1]} (EVEN) → Assigned: Custom RNN")
print("=" * 60)

# ─────────────────────────────────────────────────────────────
# SECTION 1: DATA LOADING (REAL DATASETS — NO SYNTHETIC DATA)
# ─────────────────────────────────────────────────────────────

# --- Dataset 1: Electric Production (sir's dataset) ----------
df_elec = pd.read_csv('/content/Electric_Production.csv')
elec_values = df_elec['Value'].values.astype(float)
print(f"\n  Dataset 1 loaded — Electric Production")
print(f"  Rows: {len(elec_values)}  |  Range: {elec_values.min():.2f} – {elec_values.max():.2f}")

# --- Dataset 2: International Airline Passengers (my choice) --
df_air = pd.read_csv('/content/airline-passengers.csv')
air_values = df_air['Passengers'].values.astype(float)
# WHY this dataset: Different domain (aviation vs electricity), different scale,
# strong exponential growth + seasonal trend — tests model generalisation.
print(f"\n  Dataset 2 loaded — International Airline Passengers")
print(f"  Rows: {len(air_values)}  |  Range: {air_values.min():.0f} – {air_values.max():.0f}")

# ─────────────────────────────────────────────────────────────
# SECTION 2: WINDOWING
# ─────────────────────────────────────────────────────────────

def create_windows(series, window_size, horizon):
    """
    Converts a raw time series into supervised (X, y) pairs
    by sliding a fixed window across the series.

    WHY windowing: Neural networks need fixed-size inputs.
    A window of size 9 means: given the last 9 months of data,
    predict the next 1 month.

    Example (window_size=9, horizon=1):
        series = [v0, v1, v2, ..., v396]
        X[0]   = [v0, v1, v2, v3, v4, v5, v6, v7, v8]  →  y[0] = v9
        X[1]   = [v1, v2, v3, v4, v5, v6, v7, v8, v9]  →  y[1] = v10

    WHY window_size=9: Derived from roll number. For monthly data,
    9 months captures most of a year's seasonal pattern without
    being too short (losing context) or too long (overfitting).
    """
    X, y = [], []
    for i in range(len(series) - window_size - horizon + 1):
        X.append(series[i : i + window_size])
        y.append(series[i + window_size : i + window_size + horizon])
    return np.array(X), np.array(y)


def prepare_dataset(values, window_size, horizon, train_ratio=0.8):
    """
    Scale → window → chronological split → tensors.

    WHY MinMaxScaler: Normalises values to [0, 1]. Neural networks
    train poorly on raw values of very different magnitudes. Scaling
    makes gradients stable and training faster.

    WHY chronological split (no shuffle): Time series has temporal
    dependency. Shuffling would leak future data into training,
    making test metrics unrealistically good. We always train on
    the past and test on the future — exactly as in real deployment.
    """
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(values.reshape(-1, 1)).flatten()

    X, y = create_windows(scaled, window_size, horizon)

    split   = int(len(X) * train_ratio)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # WHY unsqueeze(-1): PyTorch RNN expects input shape
    # (batch, seq_len, input_size). input_size=1 because each
    # timestep is a single scalar value.
    X_train = torch.FloatTensor(X_train).unsqueeze(-1)
    X_test  = torch.FloatTensor(X_test).unsqueeze(-1)
    y_train = torch.FloatTensor(y_train)
    y_test  = torch.FloatTensor(y_test)

    return X_train, X_test, y_train, y_test, scaler

# ─────────────────────────────────────────────────────────────
# SECTION 3: MLP BASELINE
# ─────────────────────────────────────────────────────────────

class MLPBaseline(nn.Module):
    """
    Multi-Layer Perceptron — no sequence awareness.

    WHY flatten: MLP treats the entire window as a flat vector.
    It cannot distinguish the ORDER of values — [v1,v2,v3] and
    [v3,v2,v1] produce the same output. This is the key weakness
    the Custom RNN overcomes.

    WHY include MLP at all: It is the baseline. If our RNN cannot
    beat this, our sequential model adds no real value.

    WHY ReLU: Introduces non-linearity. Without it, stacking linear
    layers is mathematically identical to a single linear layer.
    """
    def __init__(self, window_size, hidden_size, horizon):
        super().__init__()
        self.flatten = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(window_size, hidden_size * 2),
            nn.ReLU(),
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, horizon)
        )

    def forward(self, x):
        # WHY flatten: MLP needs a 1D feature vector per sample.
        # All temporal ordering is discarded here.
        x = self.flatten(x)
        return self.net(x)

# ─────────────────────────────────────────────────────────────
# SECTION 4: CUSTOM RNN — BUILT FROM SCRATCH
# ─────────────────────────────────────────────────────────────

class CustomRNNCell(nn.Module):
    """
    Single RNN cell — implements the recurrence equation manually.
    nn.RNN / nn.GRU / nn.LSTM are NOT used anywhere in this project.

    The RNN equation:
        h_t = tanh( W_xh * x_t  +  W_hh * h_{t-1}  +  b )

    WHY two weight matrices:
        W_xh maps current input to hidden space.
        W_hh maps previous hidden state to hidden space.
        Keeping them separate lets the model independently control
        how much it weights new input vs old memory.

    WHY tanh:
        Squashes output to [-1, 1], keeping the hidden state bounded
        across many timesteps. Without it, repeated matrix multiplication
        causes hidden state to explode or vanish during training.

    WHY hidden_size=14:
        Derived from roll number. 14 dimensions balances expressiveness
        and overfitting risk for datasets of this size.
    """
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.W_xh = nn.Linear(input_size, hidden_size, bias=False)
        self.W_hh = nn.Linear(hidden_size, hidden_size, bias=True)

    def forward(self, x_t, h_prev):
        """
        x_t   : (batch, input_size)  — current timestep value
        h_prev: (batch, hidden_size) — memory from previous step
        Returns h_t: updated hidden state (batch, hidden_size)
        """
        return torch.tanh(self.W_xh(x_t) + self.W_hh(h_prev))


class CustomRNN(nn.Module):
    """
    Full Custom RNN — manually unrolls CustomRNNCell over the window.

    How input moves through the model (window_size=9, hidden_size=14):
        Input : x of shape (batch, 9, 1)
        Step 0: h_0 = zeros(batch, 14)             ← no memory yet
        Step 1: h_1 = tanh(W_xh*x[:,0,:] + W_hh*h_0)
        Step 2: h_2 = tanh(W_xh*x[:,1,:] + W_hh*h_1)
        ...
        Step 9: h_9 = tanh(W_xh*x[:,8,:] + W_hh*h_8)
        Output: fc(h_9) → shape (batch, 1)

    WHY h_0 = zeros: At the start of each sequence there is no prior
    context. Zero initialisation is the standard neutral starting point.

    WHY use only final hidden state h_9:
        h_9 has been updated through all 9 timesteps. It encodes the
        full temporal context of the window — it is the model's
        compressed summary of the past 9 months.
    """
    def __init__(self, input_size, hidden_size, horizon):
        super().__init__()
        self.hidden_size = hidden_size
        self.rnn_cell    = CustomRNNCell(input_size, hidden_size)
        self.fc          = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        batch_size = x.size(0)
        seq_len    = x.size(1)

        h = torch.zeros(batch_size, self.hidden_size, device=x.device)

        # Manually unroll through each timestep in the window
        for t in range(seq_len):
            x_t = x[:, t, :]
            h   = self.rnn_cell(x_t, h)

        return self.fc(h)

# ─────────────────────────────────────────────────────────────
# SECTION 5: PREBUILT LSTM & TRANSFORMER (for comparison only)
# ─────────────────────────────────────────────────────────────

class LSTMModel(nn.Module):
    """
    WHY LSTM for comparison:
        LSTM adds three gates (input, forget, output) to the basic RNN.
        The forget gate selectively discards old memory; the input gate
        controls what new info to write. This solves the vanishing gradient
        problem that plain RNN suffers. We compare to see what we gain/lose
        by using the simpler Custom RNN.
    """
    def __init__(self, input_size, hidden_size, horizon):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc   = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


class TransformerModel(nn.Module):
    """
    WHY Transformer for comparison:
        Uses self-attention — every timestep attends to every other
        simultaneously. No sequential bottleneck. However for short
        windows (size=9) and small datasets it can overfit due to
        its larger parameter count.
    """
    def __init__(self, input_size, hidden_size, horizon, window_size):
        super().__init__()
        d_model = hidden_size if hidden_size % 2 == 0 else hidden_size + 1
        self.input_proj  = nn.Linear(input_size, d_model)
        encoder_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=2, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc          = nn.Linear(d_model * window_size, horizon)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.transformer(x)
        x = x.flatten(1)
        return self.fc(x)

# ─────────────────────────────────────────────────────────────
# SECTION 6: TRAINING
# ─────────────────────────────────────────────────────────────

def train_model(model, X_train, y_train, epochs=150, lr=1e-3):
    """
    WHY MSE loss:
        Squares the error before averaging. Large errors are penalised
        much more than small ones. In forecasting, a large miss is far
        more costly than a small miss, so MSE is appropriate.

    WHY Adam optimiser:
        Adapts learning rate per parameter using gradient moment estimates.
        Converges faster than plain SGD on non-stationary loss surfaces
        typical in time-series training.

    WHY gradient clipping at 1.0:
        During backpropagation-through-time, gradients can explode as they
        multiply across 9 timesteps. Clipping prevents destabilisingly large
        parameter updates — especially important for plain RNN with no gating.
    """
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses    = []

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        pred = model(X_train)
        loss = criterion(pred, y_train)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())

        if (epoch + 1) % 30 == 0:
            print(f"    Epoch {epoch+1:3d}/{epochs}  MSE Loss: {loss.item():.6f}")

    return losses

# ─────────────────────────────────────────────────────────────
# SECTION 7: EVALUATION
# ─────────────────────────────────────────────────────────────

def evaluate(model, X_test, y_test, scaler):
    """
    WHY inverse transform before metrics:
        Metrics on normalised [0,1] values are uninterpretable.
        After inverse transform, RMSE is in original units —
        directly meaningful (e.g. ±X units of production index).
    """
    model.eval()
    with torch.no_grad():
        preds   = model(X_test).numpy()
        actuals = y_test.numpy()

    preds_inv   = scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
    actuals_inv = scaler.inverse_transform(actuals.reshape(-1, 1)).flatten()

    mse  = mean_squared_error(actuals_inv, preds_inv)
    mae  = mean_absolute_error(actuals_inv, preds_inv)
    rmse = math.sqrt(mse)
    return mse, mae, rmse, preds_inv, actuals_inv

# ─────────────────────────────────────────────────────────────
# SECTION 8: FULL PIPELINE RUNNER
# ─────────────────────────────────────────────────────────────

COLOURS = {
    "MLP"        : "#e74c3c",
    "CustomRNN"  : "#2980b9",
    "LSTM"       : "#27ae60",
    "Transformer": "#8e44ad"
}

def run_pipeline(dataset_name, values, window_size, horizon, hidden_size):
    print(f"\n{'='*60}")
    print(f"  DATASET : {dataset_name}")
    print(f"  window_size={window_size}  |  horizon={horizon}  |  hidden_size={hidden_size}")
    print(f"{'='*60}")

    X_train, X_test, y_train, y_test, scaler = prepare_dataset(
        values, window_size, horizon
    )

    print(f"  Total windows : {len(X_train) + len(X_test)}")
    print(f"  Train samples : {len(X_train)}")
    print(f"  Test  samples : {len(X_test)}")
    print(f"  Input shape   : {tuple(X_train.shape)}  (batch, seq_len, features)")
    print(f"  Target shape  : {tuple(y_train.shape)}  (batch, horizon)")

    results    = {}
    all_losses = {}

    models = {
        "MLP"        : MLPBaseline(window_size, hidden_size, horizon),
        "CustomRNN"  : CustomRNN(1, hidden_size, horizon),
        "LSTM"       : LSTMModel(1, hidden_size, horizon),
        "Transformer": TransformerModel(1, hidden_size, horizon, window_size),
    }

    for name, model in models.items():
        print(f"\n  ── Training {name} ──")
        losses = train_model(model, X_train, y_train, epochs=150)
        mse, mae, rmse, preds, actuals = evaluate(model, X_test, y_test, scaler)
        results[name]    = (mse, mae, rmse, preds, actuals)
        all_losses[name] = losses
        print(f"    → MSE={mse:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}")

    return results, all_losses

# ─────────────────────────────────────────────────────────────
# SECTION 9: ABLATION STUDY
# ─────────────────────────────────────────────────────────────

def run_ablation(dataset_name, values, base_window, horizon, hidden_size):
    """
    WHY ablation on window size:
        window_size is our key design choice derived from the roll number.
        We need to empirically justify it — does window=9 actually perform
        better than half (4) or double (18)?
        If halving collapses performance → model needs longer context.
        If doubling barely helps → we are already at the sweet spot.
    """
    print(f"\n{'='*60}")
    print(f"  ABLATION — {dataset_name}")
    print(f"  Comparing: half={base_window//2}, original={base_window}, double={base_window*2}")
    print(f"{'='*60}")

    ablation = {}
    configs  = [
        ("half",     max(base_window // 2, 2)),
        ("original", base_window),
        ("double",   base_window * 2),
    ]

    for label, ws in configs:
        X_train, X_test, y_train, y_test, scaler = prepare_dataset(
            values, ws, horizon
        )
        model  = CustomRNN(1, hidden_size, horizon)
        losses = train_model(model, X_train, y_train, epochs=150)
        mse, mae, rmse, preds, actuals = evaluate(model, X_test, y_test, scaler)
        ablation[label] = {
            "ws": ws, "mse": mse, "mae": mae, "rmse": rmse,
            "preds": preds, "actuals": actuals, "losses": losses
        }
        print(f"  window={ws:3d} ({label:8s})  →  "
              f"MSE={mse:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}")

    return ablation

# ─────────────────────────────────────────────────────────────
# SECTION 10: PLOTTING
# ─────────────────────────────────────────────────────────────

def plot_all(results_elec, losses_elec,
             results_air,  losses_air,
             ablation_elec, ablation_air,
             save_dir="/content/plots"):

    os.makedirs(save_dir, exist_ok=True)
    plt.rcParams.update({"font.size": 10, "figure.dpi": 150})

    # ── Plot 1: Training loss — Electricity ───────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("Training Loss — Electric Production Dataset",
                 fontsize=14, fontweight="bold")
    for ax, (name, losses) in zip(axes.flatten(), losses_elec.items()):
        ax.plot(losses, color=COLOURS[name], linewidth=1.5)
        ax.set_title(name, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
        ax.grid(True, alpha=0.3)
        ax.annotate(f"Final: {losses[-1]:.5f}",
                    xy=(len(losses)-1, losses[-1]),
                    xytext=(-80, 10), textcoords="offset points",
                    fontsize=9, color=COLOURS[name])
    plt.tight_layout()
    plt.savefig(f"{save_dir}/01_training_loss_electricity.png", bbox_inches="tight")
    plt.close()
    print("  Saved: 01_training_loss_electricity.png")

    # ── Plot 2: Prediction vs Actual — Electricity ────────────────
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("Prediction vs Actual — Electric Production",
                 fontsize=14, fontweight="bold")
    for ax, (name, (mse, mae, rmse, preds, actuals)) in \
            zip(axes.flatten(), results_elec.items()):
        ax.plot(actuals, label="Actual",    color="grey",        linewidth=1.5)
        ax.plot(preds,   label="Predicted", color=COLOURS[name], linewidth=1.5, linestyle="--")
        ax.set_title(f"{name}  (RMSE={rmse:.3f})", fontweight="bold")
        ax.legend(); ax.set_xlabel("Timestep"); ax.set_ylabel("Production Index")
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/02_predictions_electricity.png", bbox_inches="tight")
    plt.close()
    print("  Saved: 02_predictions_electricity.png")

    # ── Plot 3: Model comparison — Electricity ────────────────────
    names = list(results_elec.keys())
    rmses = [results_elec[n][2] for n in names]
    maes  = [results_elec[n][1] for n in names]
    x     = np.arange(len(names))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Model Comparison — Electric Production", fontsize=13, fontweight="bold")
    for ax, vals, ylabel, title in [
        (ax1, rmses, "RMSE", "RMSE (lower = better)"),
        (ax2, maes,  "MAE",  "MAE  (lower = better)"),
    ]:
        bars = ax.bar(x, vals, color=[COLOURS[n] for n in names], alpha=0.85)
        ax.set_xticks(x); ax.set_xticklabels(names)
        ax.set_ylabel(ylabel); ax.set_title(title)
        ax.grid(True, alpha=0.3, axis="y")
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(vals)*0.01,
                    f"{v:.3f}", ha="center", va="bottom", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/03_model_comparison_electricity.png", bbox_inches="tight")
    plt.close()
    print("  Saved: 03_model_comparison_electricity.png")

    # ── Plot 4: Training loss — Air Passengers ────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("Training Loss — Airline Passengers Dataset",
                 fontsize=14, fontweight="bold")
    for ax, (name, losses) in zip(axes.flatten(), losses_air.items()):
        ax.plot(losses, color=COLOURS[name], linewidth=1.5)
        ax.set_title(name, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
        ax.grid(True, alpha=0.3)
        ax.annotate(f"Final: {losses[-1]:.5f}",
                    xy=(len(losses)-1, losses[-1]),
                    xytext=(-80, 10), textcoords="offset points",
                    fontsize=9, color=COLOURS[name])
    plt.tight_layout()
    plt.savefig(f"{save_dir}/04_training_loss_air.png", bbox_inches="tight")
    plt.close()
    print("  Saved: 04_training_loss_air.png")

    # ── Plot 5: Prediction vs Actual — Air Passengers ─────────────
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("Prediction vs Actual — Airline Passengers",
                 fontsize=14, fontweight="bold")
    for ax, (name, (mse, mae, rmse, preds, actuals)) in \
            zip(axes.flatten(), results_air.items()):
        ax.plot(actuals, label="Actual",    color="grey",        linewidth=1.5)
        ax.plot(preds,   label="Predicted", color=COLOURS[name], linewidth=1.5, linestyle="--")
        ax.set_title(f"{name}  (RMSE={rmse:.3f})", fontweight="bold")
        ax.legend(); ax.set_xlabel("Timestep"); ax.set_ylabel("Passengers (thousands)")
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/05_predictions_air.png", bbox_inches="tight")
    plt.close()
    print("  Saved: 05_predictions_air.png")

    # ── Plot 6: Model comparison — Air Passengers ─────────────────
    names = list(results_air.keys())
    rmses = [results_air[n][2] for n in names]
    maes  = [results_air[n][1] for n in names]
    x     = np.arange(len(names))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Model Comparison — Airline Passengers", fontsize=13, fontweight="bold")
    for ax, vals, ylabel, title in [
        (ax1, rmses, "RMSE", "RMSE (lower = better)"),
        (ax2, maes,  "MAE",  "MAE  (lower = better)"),
    ]:
        bars = ax.bar(x, vals, color=[COLOURS[n] for n in names], alpha=0.85)
        ax.set_xticks(x); ax.set_xticklabels(names)
        ax.set_ylabel(ylabel); ax.set_title(title)
        ax.grid(True, alpha=0.3, axis="y")
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(vals)*0.01,
                    f"{v:.3f}", ha="center", va="bottom", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/06_model_comparison_air.png", bbox_inches="tight")
    plt.close()
    print("  Saved: 06_model_comparison_air.png")

    # ── Plot 7: Ablation — Electricity ────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Ablation Study — Custom RNN Window Size (Electric Production)",
                 fontsize=13, fontweight="bold")
    for ax, label in zip(axes, ["half", "original", "double"]):
        r = ablation_elec[label]
        ax.plot(r["actuals"], color="grey",    linewidth=1.5, label="Actual")
        ax.plot(r["preds"],   color="#2980b9", linewidth=1.5,
                linestyle="--", label="Predicted")
        ax.set_title(f"window={r['ws']} ({label})\n"
                     f"RMSE={r['rmse']:.3f}  MAE={r['mae']:.3f}", fontweight="bold")
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        ax.set_xlabel("Timestep"); ax.set_ylabel("Production Index")
    plt.tight_layout()
    plt.savefig(f"{save_dir}/07_ablation_electricity.png", bbox_inches="tight")
    plt.close()
    print("  Saved: 07_ablation_electricity.png")

    # ── Plot 8: Ablation — Air Passengers ────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Ablation Study — Custom RNN Window Size (Airline Passengers)",
                 fontsize=13, fontweight="bold")
    for ax, label in zip(axes, ["half", "original", "double"]):
        r = ablation_air[label]
        ax.plot(r["actuals"], color="grey",    linewidth=1.5, label="Actual")
        ax.plot(r["preds"],   color="#2980b9", linewidth=1.5,
                linestyle="--", label="Predicted")
        ax.set_title(f"window={r['ws']} ({label})\n"
                     f"RMSE={r['rmse']:.3f}  MAE={r['mae']:.3f}", fontweight="bold")
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        ax.set_xlabel("Timestep"); ax.set_ylabel("Passengers (thousands)")
    plt.tight_layout()
    plt.savefig(f"{save_dir}/08_ablation_air.png", bbox_inches="tight")
    plt.close()
    print("  Saved: 08_ablation_air.png")

    # ── Plot 9: Residual error analysis ──────────────────────────
    fig, axes = plt.subplots(2, 4, figsize=(20, 9))
    fig.suptitle("Residual Error Analysis — Where Each Model Fails",
                 fontsize=13, fontweight="bold")
    for col, (name, (mse, mae, rmse, preds, actuals)) in \
            enumerate(results_elec.items()):
        ax = axes[0, col]
        residuals = actuals - preds
        ax.plot(residuals, color=COLOURS[name], linewidth=0.8, alpha=0.8)
        ax.axhline(0, color="black", linewidth=1, linestyle="--")
        threshold = 2 * np.std(residuals)
        ax.fill_between(range(len(residuals)), residuals, 0,
                        where=(np.abs(residuals) > threshold),
                        alpha=0.35, color="red", label="|err|>2σ")
        ax.set_title(f"{name} — Electricity", fontweight="bold", fontsize=9)
        ax.set_xlabel("Timestep"); ax.set_ylabel("Actual − Predicted")
        ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    for col, (name, (mse, mae, rmse, preds, actuals)) in \
            enumerate(results_air.items()):
        ax = axes[1, col]
        residuals = actuals - preds
        ax.plot(residuals, color=COLOURS[name], linewidth=0.8, alpha=0.8)
        ax.axhline(0, color="black", linewidth=1, linestyle="--")
        threshold = 2 * np.std(residuals)
        ax.fill_between(range(len(residuals)), residuals, 0,
                        where=(np.abs(residuals) > threshold),
                        alpha=0.35, color="red", label="|err|>2σ")
        ax.set_title(f"{name} — Air Passengers", fontweight="bold", fontsize=9)
        ax.set_xlabel("Timestep"); ax.set_ylabel("Actual − Predicted")
        ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{save_dir}/09_residual_analysis.png", bbox_inches="tight")
    plt.close()
    print("  Saved: 09_residual_analysis.png")

    print(f"\n  All 9 plots saved to: {save_dir}/")

# ─────────────────────────────────────────────────────────────
# SECTION 11: MAIN
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # Dataset 1: Electric Production (sir's dataset)
    results_elec, losses_elec = run_pipeline(
        "Electric Production (Sir's Dataset)",
        elec_values, window_size, prediction_horizon, hidden_size
    )

    # Dataset 2: Airline Passengers (my choice)
    results_air, losses_air = run_pipeline(
        "International Airline Passengers (My Choice)",
        air_values, window_size, prediction_horizon, hidden_size
    )

    # Ablation on both datasets
    ablation_elec = run_ablation(
        "Electric Production", elec_values,
        window_size, prediction_horizon, hidden_size
    )
    ablation_air = run_ablation(
        "Airline Passengers", air_values,
        window_size, prediction_horizon, hidden_size
    )

    # Generate all 9 plots
    print("\n  Generating all plots...")
    plot_all(results_elec, losses_elec,
             results_air,  losses_air,
             ablation_elec, ablation_air)

    # ── Final Summary ─────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("  FINAL RESULTS — ELECTRIC PRODUCTION")
    print(f"{'='*60}")
    print(f"  {'Model':15s}  {'MSE':>10s}  {'MAE':>10s}  {'RMSE':>10s}")
    print(f"  {'-'*52}")
    for name, (mse, mae, rmse, _, __) in results_elec.items():
        print(f"  {name:15s}  {mse:10.4f}  {mae:10.4f}  {rmse:10.4f}")

    print(f"\n{'='*60}")
    print("  FINAL RESULTS — AIRLINE PASSENGERS")
    print(f"{'='*60}")
    print(f"  {'Model':15s}  {'MSE':>10s}  {'MAE':>10s}  {'RMSE':>10s}")
    print(f"  {'-'*52}")
    for name, (mse, mae, rmse, _, __) in results_air.items():
        print(f"  {name:15s}  {mse:10.4f}  {mae:10.4f}  {rmse:10.4f}")

    print(f"\n{'='*60}")
    print("  ABLATION — ELECTRIC PRODUCTION (Custom RNN)")
    print(f"{'='*60}")
    for label, r in ablation_elec.items():
        print(f"  {label:8s} (ws={r['ws']:2d})  →  RMSE={r['rmse']:.4f}  MAE={r['mae']:.4f}")

    print(f"\n{'='*60}")
    print("  ABLATION — AIRLINE PASSENGERS (Custom RNN)")
    print(f"{'='*60}")
    for label, r in ablation_air.items():
        print(f"  {label:8s} (ws={r['ws']:2d})  →  RMSE={r['rmse']:.4f}  MAE={r['mae']:.4f}")

    print(f"\n  Roll Number : {ROLL_NUMBER}")
    print(f"  window_size={window_size}  |  horizon={prediction_horizon}  |  hidden_size={hidden_size}")
    print(f"  Model: Custom RNN (built from scratch, no nn.RNN used)")

  ROLL NUMBER PARAMETER DERIVATION
  Roll Number        : 102303372
  Digits             : [1, 0, 2, 3, 0, 3, 3, 7, 2]
  Sum of digits      : 21
  Last 2 digits      : 72
  First 3 digits     : 102
  window_size        = 21 mod 10 + 8 = 9
  prediction_horizon = 72 mod 3 + 1 = 1
  hidden_size        = 102 mod 16 + 8 = 14
  Last digit = 2 (EVEN) → Assigned: Custom RNN

  Dataset 1 loaded — Electric Production
  Rows: 397  |  Range: 55.32 – 129.40

  Dataset 2 loaded — International Airline Passengers
  Rows: 144  |  Range: 104 – 622

  DATASET : Electric Production (Sir's Dataset)
  window_size=9  |  horizon=1  |  hidden_size=14
  Total windows : 388
  Train samples : 310
  Test  samples : 78
  Input shape   : (310, 9, 1)  (batch, seq_len, features)
  Target shape  : (310, 1)  (batch, horizon)

  ── Training MLP ──
    Epoch  30/150  MSE Loss: 0.074774
    Epoch  60/150  MSE Loss: 0.038675
    Epoch  90/150  MSE Loss: 0.025662
    Epoch 120/150  MSE Loss: 0.021781
    Epoch 150/150  MSE 